# collate-gex-metadata
Assemble one row of metadata per RNA-seq library we analyze, from S.T.2 biosamples, for SJ and CBTN.

## Requirements
- `data/Supplementary Tables.xlsx` (S.T.2)
- `data/source/sjcloud/SAMPLE_INFO_20260901.txt`
- Expression matrices from `0-generate-gex-matrices.ipynb`
- env `py3.yml`

## Results

PBTA: RNA available for 1667 / 2366 samples  
SJC: RNA available for 1028 / 1230 samples  
Total: RNA available for 2695 / 3596 samples  

## Outputs
`data/source/gene-expression/rna-metadata.tsv`:
Metadata table, one row per RNA-seq library, with the following columns. Columns marked *SJ only* or *CBTN only* are NA for the other cohort.
- `rna_id` (index): RNA-seq library ID, as in the expression matrix columns. CBTN: Kids First biospecimen ID (`BS_`); SJ: biosample name.
- `cohort`: `SJC` or `CBTN`. Also determines the quantification: SJ HTSeq counts, CBTN RSEM expected counts.
- `patient_id`: patient ID, from S.T.2.
- `event_id`: collection event, linking the library to S.T.2 WGS biosamples. CBTN: 7316- sample ID (S.T.2 `external_sample_id`); SJ: the biosample itself.
- `rna_selection`: `PolyA` or `Total` (ribosomal depletion).
- `strandedness`: `Stranded` or `Unstranded`. All CBTN libraries are stranded.
- `read_length`: *SJ only.* Read length per read, collapsed to `101` (100-101) or `126` (125-126); other values NA.
- `seq_center`: *CBTN only.* Sequencing center: `NantOmics`, `BGI@CHOP` or `Ashion`; NA is mostly the X01 batch.
- `cancer_type`: from S.T.2.
- `cancer_subclass`: from S.T.2. Where WGS biosamples from the event disagree, taken from the representative biosample (`in_unique_tumor_set`).
- `tumor_history`: from S.T.2, eg. Diagnosis, Progressive, Autopsy. Representative biosample where the event disagrees.
- `sex`: from S.T.2.
- `age_at_collection`: *CBTN only.* Age at collection of the event, in days.
- `age_at_diagnosis`: age at the patient's earliest evidence of disease, in days.
- `ecDNA_any`: `True` if ecDNA was detected in any WGS biosample from the event.

In [ ]:
import pandas as pd
import sys
sys.path.append('../../src')
import data_imports
import gex_imports
pd.set_option('display.max_rows', 100)

biosamples = data_imports.import_biosamples()

In [ ]:
def summarize(biosamples):
    PBTA = ['OpenPBTA','PBTA-X01','PNOC']
    a = len(biosamples[biosamples.rna_available & biosamples.cohort.isin(PBTA)])
    b = len(biosamples[biosamples.cohort.isin(PBTA)])
    c = len(biosamples[biosamples.rna_available & ~biosamples.cohort.isin(PBTA)])
    d = len(biosamples[~biosamples.cohort.isin(PBTA)])
    print(f'PBTA: RNA available for {a} / {b} samples  ')
    print(f'SJC: RNA available for {c} / {d} samples  ')
    print(f'Total: RNA available for {a+c} / {b+d} samples  ')
summarize(biosamples)

## SJ RNAseq
SJ RNA-seq libraries by libary prep, strandedness and read length:
```
rna_selection  strandedness  read_length     ct
PolyA          Unstranded    101            153
                             <NA>             1
Total          Stranded      101             29
                             126            830
                             <NA>            10
NaN            NaN           <NA>             1
```
Notes:
- Re-acquired 32 count files which were expected to be here but weren't for some reason;
- 'Stranded' is stranded-reverse; dropped reverse since this info not available for CBTN;
- Dropped 8 samples annotated as FFPE;
- 9 SJACT samples get new patient IDs in 2026;
- Strandedness imputed from library prep for 8 samples;
- ambiguous, erroneous or singleton read lengths dropped for 9 samples;

In [ ]:
sj_rna = gex_imports.import_sj_rna_metadata(biosamples)
print(f"SJ RNA-seq libraries: {len(sj_rna)}")
sj_rna.head()

In [ ]:
# SJ RNA-seq libraries by libary prep, strandedness and read length:
sj_rna.groupby(['rna_selection','strandedness','read_length'],dropna=False).size()

## CBTN RNA-seq
CBTN RNA-seq libraries by libary prep, strandedness and sequencing center:
```
rna_selection  strandedness  seq_center     ct
PolyA          Stranded      BGI@CHOP       14
                             NaN           696
Total          Stranded      Ashion         12
                             BGI@CHOP       36
                             NantOmics     810
```
Notes:
- Drop 2 exome-capture libraries;
- Drop singleton libraries from TTGRI and Broad;
- NA mostly correspond to X01 dataset, presumably sequenced at the same center(?)
- Read length annotations not readily available

In [ ]:
cbtn_rna = gex_imports.import_cbtn_rna_metadata(biosamples)
print(f"CBTN RNA-seq libraries: {len(cbtn_rna)}")
cbtn_rna.head()

In [ ]:
# CBTN RNA-seq libraries by libary prep, strandedness and read length:
cbtn_rna.groupby(['rna_selection','strandedness','seq_center'],dropna=False).size()

In [ ]:
rna_tbl = gex_imports.generate_rna_metadata_table(biosamples)
rna_tbl.to_csv('../../data/source/gene-expression/rna-metadata.tsv', sep='\t')

In [ ]:
rna_tbl.head()